# Vesuvius Challenge — nnU-Net Training

Train nnU-Net v2 for 3D surface segmentation, following the 1st place solution approach.

**References:**
- [1st Place Solution Writeup](https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection/writeups/1st-place-solution-for-the-vesuvius-challenge-su)
- [Baseline Notebook (jirkaborovec)](https://www.kaggle.com/code/jirkaborovec/surface-train-inference-3d-segm-gpu-augment)
- [nnU-Net v2 GitHub](https://github.com/MIC-DKFZ/nnUNet)

In [ ]:
!pip install nnunetv2 -q

In [ ]:
import os
import csv
import json
import shutil
import time

import torch
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Configuration

In [ ]:
# --- Paths ---
DATA_DIR = '/kaggle/input/vesuvius-challenge-surface-detection'
WORK_DIR = '/kaggle/working'

NNUNET_RAW = os.path.join(WORK_DIR, 'nnUNet_raw')
NNUNET_PREPROCESSED = os.path.join(WORK_DIR, 'nnUNet_preprocessed')
NNUNET_RESULTS = os.path.join(WORK_DIR, 'nnUNet_results')

DATASET_ID = 11
DATASET_NAME = 'Dataset011_Vesuvius'

# --- Training ---
NUM_EPOCHS = 200
FOLD = 'all'               # Train on all data (no cross-validation)
CONFIGURATION = '3d_fullres'
VOXEL_SPACING = 7.91       # Isotropic voxel spacing in micrometers

# --- Set environment variables ---
os.environ['nnUNet_raw'] = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROCESSED
os.environ['nnUNet_results'] = NNUNET_RESULTS

for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

print(f'nnUNet_raw:          {NNUNET_RAW}')
print(f'nnUNet_preprocessed: {NNUNET_PREPROCESSED}')
print(f'nnUNet_results:      {NNUNET_RESULTS}')

## Register Custom 200-Epoch Trainer

nnU-Net's default trainer runs 1000 epochs. We create a subclass that sets `num_epochs = 200` and place it in the nnU-Net trainers directory so it can be found by name.

In [ ]:
import nnunetv2

trainer_code = '''import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_200epochs(nnUNetTrainer):
    def __init__(self, plans, configuration, fold, dataset_json,
                 device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 200
'''

trainer_dir = os.path.join(
    os.path.dirname(nnunetv2.__path__[0]),
    'nnunetv2', 'training', 'nnUNetTrainer', 'variants', 'training_length'
)
trainer_path = os.path.join(trainer_dir, 'nnUNetTrainer_200epochs.py')
os.makedirs(trainer_dir, exist_ok=True)

with open(trainer_path, 'w') as f:
    f.write(trainer_code)

print(f'Wrote custom trainer to {trainer_path}')
print(f'Trainer name: nnUNetTrainer_200epochs')

## Convert Data to nnU-Net Format

nnU-Net expects:
- `imagesTr/{case_id}_0000.tif` + spacing JSON
- `labelsTr/{case_id}.tif` + spacing JSON
- `dataset.json`

We use symlinks to avoid copying ~42 GB of data.

In [ ]:
dataset_dir = os.path.join(NNUNET_RAW, DATASET_NAME)
images_dst = os.path.join(dataset_dir, 'imagesTr')
labels_dst = os.path.join(dataset_dir, 'labelsTr')
os.makedirs(images_dst, exist_ok=True)
os.makedirs(labels_dst, exist_ok=True)

# Read sample IDs from train.csv
csv_path = os.path.join(DATA_DIR, 'train.csv')
sample_ids = []
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sample_ids.append(row['id'])

# Exclude deprecated samples
deprecated_dir = os.path.join(DATA_DIR, 'deprecated_train_images')
deprecated_ids = set()
if os.path.isdir(deprecated_dir):
    for fn in os.listdir(deprecated_dir):
        if fn.endswith('.tif'):
            deprecated_ids.add(os.path.splitext(fn)[0])

sample_ids = sorted([s for s in sample_ids if s not in deprecated_ids])
print(f'Total training samples: {len(sample_ids)}')

def write_spacing_json(path, spacing):
    with open(path, 'w') as f:
        json.dump({'spacing': [spacing, spacing, spacing]}, f)

linked = 0
t0 = time.time()
for i, sid in enumerate(sample_ids):
    img_src = os.path.join(DATA_DIR, 'train_images', f'{sid}.tif')
    lbl_src = os.path.join(DATA_DIR, 'train_labels', f'{sid}.tif')

    if not os.path.exists(img_src) or not os.path.exists(lbl_src):
        continue

    case_id = f'vesuvius_{i:04d}'

    img_dst = os.path.join(images_dst, f'{case_id}_0000.tif')
    lbl_dst = os.path.join(labels_dst, f'{case_id}.tif')

    if not os.path.exists(img_dst):
        os.symlink(os.path.abspath(img_src), img_dst)
    if not os.path.exists(lbl_dst):
        os.symlink(os.path.abspath(lbl_src), lbl_dst)

    write_spacing_json(os.path.join(images_dst, f'{case_id}.json'), VOXEL_SPACING)
    write_spacing_json(os.path.join(labels_dst, f'{case_id}.json'), VOXEL_SPACING)

    linked += 1
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(sample_ids)}')

# Write dataset.json
dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {
        'background': 0,
        'surface': 1,
        'interior': 2
    },
    'numTraining': linked,
    'file_ending': '.tif',
}
with open(os.path.join(dataset_dir, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'\nDone: {linked} samples linked in {time.time() - t0:.1f}s')
print(f'Dataset directory: {dataset_dir}')

## Plan and Preprocess

nnU-Net auto-configures the pipeline based on dataset properties (spacing, sizes, class distribution). This step reads all training volumes to determine optimal patch size, batch size, and normalization.

In [ ]:
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -c {CONFIGURATION}

## Train

Training on all data (fold='all') with our custom 200-epoch trainer.

The `--npz` flag saves softmax probabilities alongside predictions, which is useful for ensembling later. The `--c` flag can be added to resume from a checkpoint.

**Expected time:** ~5-7 min/epoch on T4 GPU with 704 samples. Total: ~17-24 hours for 200 epochs.

In [ ]:
# First run: remove --c flag
# To resume: add --c flag
!nnUNetv2_train {DATASET_ID} {CONFIGURATION} {FOLD} \
    -tr nnUNetTrainer_200epochs \
    --npz

## Save Checkpoint

Copy the trained model to the Kaggle output directory for download. The checkpoint includes the full model state, plans, and dataset.json needed for inference.

In [ ]:
model_dir = os.path.join(
    NNUNET_RESULTS, DATASET_NAME,
    f'nnUNetTrainer_200epochs__nnUNetPlans__{CONFIGURATION}'
)

output_model_dir = os.path.join(WORK_DIR, 'nnunet_model')
os.makedirs(output_model_dir, exist_ok=True)

fold_dir = os.path.join(model_dir, f'fold_{FOLD}')

files_to_copy = [
    'checkpoint_final.pth',
    'checkpoint_best.pth',
    'training_log_*.txt',
]

# Copy fold directory
dst_fold = os.path.join(output_model_dir, f'fold_{FOLD}')
if os.path.exists(fold_dir):
    shutil.copytree(fold_dir, dst_fold, dirs_exist_ok=True)
    print(f'Copied fold directory to {dst_fold}')

# Copy plans and dataset.json (needed for inference)
for fname in ['plans.json', 'dataset.json', 'dataset_fingerprint.json']:
    src = os.path.join(model_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(output_model_dir, fname))
        print(f'Copied {fname}')

# Show sizes
total_size = 0
for root, dirs, files in os.walk(output_model_dir):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        total_size += size
        print(f'  {os.path.relpath(fpath, output_model_dir)}: {size / 1e6:.1f} MB')

print(f'\nTotal model size: {total_size / 1e6:.1f} MB')
print(f'\nModel saved to: {output_model_dir}')
print('Download this directory and upload as a Kaggle Model for submission.')

## Resume Training (if session timed out)

If the session timed out mid-training:

1. Re-run cells 1-6 (install, config, trainer, convert, preprocess)
2. Upload the previous `nnunet_model/` output as a Kaggle dataset
3. Copy checkpoint back and resume:

```python
# Copy checkpoint from uploaded dataset back to nnUNet_results
prev_model = '/kaggle/input/your-nnunet-checkpoint'
resume_dir = os.path.join(
    NNUNET_RESULTS, DATASET_NAME,
    f'nnUNetTrainer_200epochs__nnUNetPlans__{CONFIGURATION}',
    f'fold_{FOLD}'
)
os.makedirs(resume_dir, exist_ok=True)
shutil.copy2(
    os.path.join(prev_model, f'fold_{FOLD}', 'checkpoint_final.pth'),
    os.path.join(resume_dir, 'checkpoint_final.pth')
)
```

Then run the training cell with `--c` added.